# Présentation du notebook

Ce notebook présente les requêtes AQL de référence (Gold AQL) utilisées comme base de comparaison pour l'évaluation des requêtes générées automatiquement.

# Objectif

Construire un ensemble de requêtes de référence permettant l'évaluation fonctionnelle du système.

In [2]:
import json
import os
import pandas as pd
from pathlib import Path
from arango import ArangoClient

In [3]:
from pathlib import Path

# Détecter automatiquement la racine du projet
current = Path.cwd().resolve()

if (current / "arangodb").exists() and (current / "data").exists():
    # Notebook lancé depuis la racine du projet
    PROJECT_ROOT = current

elif (current.parent / "arangodb").exists() and (current.parent / "data").exists():
    # Notebook lancé depuis notebooks/
    PROJECT_ROOT = current.parent

else:
    raise FileNotFoundError(
        "Impossible de détecter la racine du projet."
    )

SCHEMA_PATH = (
    PROJECT_ROOT
    / "arangodb"
    / "schema_description.json"
)

QUESTIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "benchmark"
    / "YELP.questions.txt"
)

BENCHMARK_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "benchmark"
    / "yelp_benchmark.json"
)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SCHEMA_PATH :", SCHEMA_PATH)
print("QUESTIONS_PATH :", QUESTIONS_PATH)
print("BENCHMARK_OUTPUT_PATH :", BENCHMARK_OUTPUT_PATH)

PROJECT_ROOT : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project
SCHEMA_PATH : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\arangodb\schema_description.json
QUESTIONS_PATH : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\data\benchmark\YELP.questions.txt
BENCHMARK_OUTPUT_PATH : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\data\benchmark\yelp_benchmark.json


In [4]:
assert SCHEMA_PATH.exists(), f"Schema introuvable : {SCHEMA_PATH}"
assert QUESTIONS_PATH.exists(), f"Questions introuvables : {QUESTIONS_PATH}"
assert BENCHMARK_OUTPUT_PATH.parent.exists(), "Dossier benchmark introuvable"

print("Tous les chemins sont valides.")

Tous les chemins sont valides.


In [5]:
from arango import ArangoClient

ARANGO_URL = "http://localhost:8529"
DATABASE_NAME = "YelpDB"
USERNAME = "root"
PASSWORD = "root"   # adapte si ton mot de passe est différent

client = ArangoClient(
    hosts=ARANGO_URL,
    request_timeout=300
)

db = client.db(
    DATABASE_NAME,
    username=USERNAME,
    password=PASSWORD
)

print("Connexion à YelpDB réussie.")

Connexion à YelpDB réussie.


In [6]:
print("Collections disponibles :")

for collection in db.collections():
    if not collection["name"].startswith("_"):
        print("-", collection["name"])

Collections disponibles :
- Categories
- ReviewsBusiness
- BusinessCategory
- Tips
- BusinessNeighborhood
- Users
- TipsBusiness
- WritesReview
- BusinessCheckin
- Businesses
- Checkins
- Neighborhoods
- Reviews
- WritesTip


In [7]:
import json

with open(
    SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:
    schema = json.load(f)

print("Base :", schema["database"])
print("Graph :", schema["graph"])
print("Document collections :", len(schema["document_collections"]))
print("Edge collections :", len(schema["edge_collections"]))
print("Relations :", len(schema["relationships"]))

Base : YelpDB
Graph : YelpGraph
Document collections : 7
Edge collections : 7
Relations : 7


In [8]:
print("DOCUMENT COLLECTIONS")
for name in schema["document_collections"]:
    print("-", name)

print("\nEDGE COLLECTIONS")
for name in schema["edge_collections"]:
    print("-", name)

DOCUMENT COLLECTIONS
- Businesses
- Users
- Reviews
- Tips
- Categories
- Checkins
- Neighborhoods

EDGE COLLECTIONS
- WritesReview
- ReviewsBusiness
- WritesTip
- TipsBusiness
- BusinessCategory
- BusinessCheckin
- BusinessNeighborhood


In [9]:
with open(
    QUESTIONS_PATH,
    "r",
    encoding="utf-8"
) as f:
    questions = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Nombre de questions chargées :", len(questions))

Nombre de questions chargées : 128


In [10]:
for i, question in enumerate(questions[:10], start=1):
    print(f"Q{i:03d} - {question}")

Q001 - Give me all the moroccan restaurants in Texas
Q002 - List all the Italian restaurants in Los Angeles
Q003 - find the number of preschools in Madison
Q004 - List all the restaurants rated more than 3.5
Q005 - List all the businesses with more than 4.5 stars
Q006 - find the number of restaurants rated more than 3.5
Q007 - Which Thai restaurant has the most number of reviews
Q008 - find all cities which has a “Taj Mahal” restaurant
Q009 - find the total checkins in Moroccan restaurants in Los Angeles
Q010 - find the total checkins in Moroccan restaurants in Los Angeles on Fridays


In [12]:
assert len(questions) == 128, (
    f"Erreur : 128 questions attendues, mais {len(questions)} trouvées."
)

print("Benchmark SQLizer/Yelp chargé correctement : 128 questions.")

Benchmark SQLizer/Yelp chargé correctement : 128 questions.


In [13]:
keywords = [
    "moroccan",
    "italian",
    "restaurant",
    "thai",
    "preschool"
]

for keyword in keywords:
    print(f"\n--- {keyword.upper()} ---")

    query = """
    FOR c IN Categories
        FILTER CONTAINS(
            LOWER(c.category_name),
            LOWER(@keyword)
        )
        COLLECT category = c.category_name
        SORT category
        RETURN category
    """

    result = list(
        db.aql.execute(
            query,
            bind_vars={"keyword": keyword}
        )
    )

    print(result)


--- MOROCCAN ---
['Moroccan']

--- ITALIAN ---
['Italian']

--- RESTAURANT ---
['Restaurants']

--- THAI ---
['Thai']

--- PRESCHOOL ---
['Preschools']


In [14]:
query = """
FOR b IN Businesses
    FILTER b.state != null
    COLLECT state = b.state WITH COUNT INTO total
    SORT state
    RETURN {
        state: state,
        total: total
    }
"""

states = list(db.aql.execute(query))

for item in states:
    print(item)

{'state': 'Alabama', 'total': 1}
{'state': 'Alaska', 'total': 1}
{'state': 'Arizona', 'total': 36500}
{'state': 'Baden-Wurttemberg', 'total': 1055}
{'state': 'California', 'total': 4}
{'state': 'East Lothian', 'total': 11}
{'state': 'Fife', 'total': 5}
{'state': 'Florida', 'total': 2}
{'state': 'Illinois', 'total': 808}
{'state': 'Lothian', 'total': 3298}
{'state': 'Midlothian', 'total': 163}
{'state': 'Minnesota', 'total': 1}
{'state': 'Nevada', 'total': 23591}
{'state': 'New Mexico', 'total': 1}
{'state': 'North Carolina', 'total': 6835}
{'state': 'North Rhine-Westphalia', 'total': 1}
{'state': 'Ontario', 'total': 530}
{'state': 'Pennsylvania', 'total': 4086}
{'state': 'Quebec', 'total': 5591}
{'state': 'Rhineland-Palatinate', 'total': 18}
{'state': 'South Carolina', 'total': 325}
{'state': 'Tamaulipas', 'total': 1}
{'state': 'Texas', 'total': 3}
{'state': 'West Hampshire', 'total': 1}
{'state': 'Wisconsin', 'total': 3066}


In [15]:
reference_queries = [
    {
        "id": "Q001",
        "question": questions[0],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q002",
        "question": questions[1],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Italian"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q003",
        "question": questions[2],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories"
        ],
        "aql": """
LET preschools = (
    FOR b IN Businesses
        FILTER b.city == "Madison"

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Preschools"

            RETURN DISTINCT b._id
)

RETURN LENGTH(preschools)
"""
    },

    {
        "id": "Q004",
        "question": questions[3],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.rating > 3.5

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q005",
        "question": questions[4],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": [
            "Businesses"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.rating > 4.5

    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        state: b.state,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q006",
        "question": questions[5],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories"
        ],
        "aql": """
LET restaurants = (
    FOR b IN Businesses
        FILTER b.rating > 3.5

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Restaurants"

            RETURN DISTINCT b._id
)

RETURN LENGTH(restaurants)
"""
    },

    {
        "id": "Q007",
        "question": questions[6],
        "difficulty": "medium",
        "type": "graph_sort_limit",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories"
        ],
        "aql": """
FOR b IN Businesses

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Thai"

        SORT b.review_count DESC
        LIMIT 1

        RETURN {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            review_count: b.review_count,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q008",
        "question": questions[7],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": [
            "Businesses"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Taj Mahal"

    RETURN DISTINCT b.city
"""
    }
]

print(f"Nombre de Gold AQL préparées : {len(reference_queries)}")

Nombre de Gold AQL préparées : 8


In [16]:
for ref in reference_queries:
    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))
        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q001 - Give me all the moroccan restaurants in Texas
OK
Nombre de résultats : 0
Exemple : []
Q002 - List all the Italian restaurants in Los Angeles
OK
Nombre de résultats : 0
Exemple : []
Q003 - find the number of preschools in Madison
OK
Nombre de résultats : 1
Exemple : [2]
Q004 - List all the restaurants rated more than 3.5
OK
Nombre de résultats : 10605
Exemple : [{'business_id': 'vvPzcOhbQn5fLQUAIxcP6A', 'name': "Deluca's Diner", 'city': 'Pittsburgh', 'rating': 4}, {'business_id': 'cjD2yGRhT5yaSj_KP55Ptw', 'name': 'Kaya', 'city': 'Pittsburgh', 'rating': 4}, {'business_id': 'w7VSMULJ9GtRi3CBRxoJtg', 'name': 'Robert Wholey and Co Fish Market', 'city': 'Pittsburgh', 'rating': 4}]
Q005 - List all the businesses with more than 4.5 stars
OK
Nombre de résultats : 12964
Exemple : [{'business_id': 'bRijcLYfpRdxaABwVbeJ_A', 'name': 'Ziltzer Vivian Gee, MD', 'city': 'Scottsdale', 'state': 'Arizona', 'rating': 5}, {'business_id': '7K9SJDy5Y0iLtCBCKbvS0w', 'name': "Commander's Palace", 'city':

In [17]:
diagnostic_queries = {
    "Moroccan_businesses": """
        FOR b IN Businesses
            FOR c IN 1..1 OUTBOUND b BusinessCategory
                FILTER c.category_name == "Moroccan"
                RETURN DISTINCT {
                    name: b.name,
                    city: b.city,
                    state: b.state,
                    rating: b.rating
                }
    """,

    "Italian_Los_Angeles": """
        FOR b IN Businesses
            FILTER CONTAINS(LOWER(b.city), "los angeles")

            FOR c IN 1..1 OUTBOUND b BusinessCategory
                FILTER c.category_name == "Italian"

                RETURN DISTINCT {
                    name: b.name,
                    city: b.city,
                    state: b.state,
                    rating: b.rating
                }
    """,

    "Los_Angeles_variants": """
        FOR b IN Businesses
            FILTER CONTAINS(LOWER(b.city), "los")
            COLLECT city = b.city WITH COUNT INTO total
            SORT total DESC
            RETURN {
                city: city,
                total: total
            }
    """
}

for name, query in diagnostic_queries.items():
    print("\\n", "=" * 60)
    print(name)
    print("=" * 60)

    result = list(db.aql.execute(query))

    print("Nombre :", len(result))
    print("Exemple :", result[:20])

\n ============================================================
Moroccan_businesses
Nombre : 26
Exemple : [{'name': "Lulu's Deli and Restaurant", 'city': 'Madison', 'state': 'Wisconsin', 'rating': 3.5}, {'name': 'La Khaima', 'city': 'Montréal', 'state': 'Quebec', 'rating': 4}, {'name': 'Salon Mogador', 'city': 'Montréal', 'state': 'Quebec', 'rating': 4}, {'name': 'Restaurant Au-Tarot', 'city': 'Montréal', 'state': 'Quebec', 'rating': 4}, {'name': 'Restaurant Kamela', 'city': 'Montréal', 'state': 'Quebec', 'rating': 4.5}, {'name': 'Bar Aux Trois Devins', 'city': 'Lachine', 'state': 'Quebec', 'rating': 4}, {'name': 'Casablanca Cafe', 'city': 'Charlotte', 'state': 'North Carolina', 'rating': 3.5}, {'name': 'Kous Kous Cafe', 'city': 'Pittsburgh', 'state': 'Pennsylvania', 'rating': 4.5}, {'name': 'Restaurant El Morocco', 'city': 'Montréal', 'state': 'Quebec', 'rating': 2.5}, {'name': 'Marrakesh Express', 'city': 'Henderson', 'state': 'Nevada', 'rating': 3}, {'name': "Faouzi's Restaurant", '

In [18]:
query = """
FOR c IN Checkins
    FILTER c.day != null
    COLLECT day = c.day WITH COUNT INTO total
    SORT day
    RETURN {
        day: day,
        total: total
    }
"""

days = list(db.aql.execute(query))

for item in days:
    print(item)

{'day': 'Friday', 'total': 122098}
{'day': 'Monday', 'total': 122098}
{'day': 'Saturday', 'total': 122098}
{'day': 'Sunday', 'total': 122098}
{'day': 'Thursday', 'total': 122098}
{'day': 'Tuesday', 'total': 122098}
{'day': 'Wednesday', 'total': 122098}


In [19]:
names_to_check = ["Niloofar", "Michelle", "Patrick"]

for name in names_to_check:
    query = """
    FOR u IN Users
        FILTER u.name == @name
        RETURN {
            user_id: u.user_id,
            name: u.name
        }
    """

    result = list(
        db.aql.execute(
            query,
            bind_vars={"name": name}
        )
    )

    print(f"{name} -> count = {len(result)}")
    print(result[:10])

Niloofar -> count = 2
[{'user_id': 'j6VY9UwaQCiEWfQtNyEouQ', 'name': 'Niloofar'}, {'user_id': '1Or2rEztlk4AvlOwXhj6SQ', 'name': 'Niloofar'}]
Michelle -> count = 3700
[{'user_id': 'GNThGPoZANDUwA3TdbSdFg', 'name': 'Michelle'}, {'user_id': 'XED9_Urs-pCp5tzWC_ka-w', 'name': 'Michelle'}, {'user_id': '-CqOoxshXSK6GYVfMdpIkw', 'name': 'Michelle'}, {'user_id': 'kVeyXaqwrVeWSqP-ROemfw', 'name': 'Michelle'}, {'user_id': 'AtaTUzJJc5cQAlfpkLajEw', 'name': 'Michelle'}, {'user_id': 'V8jwIxg3P-qNZ3llbAS5Mw', 'name': 'Michelle'}, {'user_id': 'g6k4mQt0EkAfgWuPc8R3yg', 'name': 'Michelle'}, {'user_id': 'Wkm6B-wEC_7uUfQl8Dfauw', 'name': 'Michelle'}, {'user_id': 'MFnB7AfI5RtsJXoG4gdnzg', 'name': 'Michelle'}, {'user_id': 'm40GCQYsohTY3U0qhJwFXg', 'name': 'Michelle'}]
Patrick -> count = 1371
[{'user_id': 'fRscdpeKUb5Z279-sWvfJQ', 'name': 'Patrick'}, {'user_id': 'PDgEwBu58cglR5-b64grug', 'name': 'Patrick'}, {'user_id': 'EScQuI8RUzPkIYCEvSXcgw', 'name': 'Patrick'}, {'user_id': 'vGeLTvEj-3bPeKAgoKMgew', 'name'

In [20]:
query = """
FOR b IN Businesses
    FILTER b.name == "Cafe Zinho"
    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        state: b.state,
        rating: b.rating
    }
"""

result = list(db.aql.execute(query))

print("Nombre de Cafe Zinho :", len(result))
print(result)

Nombre de Cafe Zinho : 1
[{'business_id': 'l_Wey2YC6NvVUIH8vTygGg', 'name': 'Cafe Zinho', 'city': 'Pittsburgh', 'state': 'Pennsylvania', 'rating': 4}]


In [21]:
reference_queries.extend([

    {
        "id": "Q009",
        "question": questions[8],
        "difficulty": "advanced",
        "type": "graph_aggregation",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories",
            "BusinessCheckin",
            "Checkins"
        ],
        "aql": """
LET values = (
    FOR b IN Businesses
        FILTER b.city == "Los Angeles"

        LET categories = (
            FOR c IN 1..1 OUTBOUND b BusinessCategory
                RETURN c.category_name
        )

        FILTER "Moroccan" IN categories
        FILTER "Restaurants" IN categories

        FOR ch IN 1..1 OUTBOUND b BusinessCheckin
            RETURN ch.count
)

RETURN SUM(values)
"""
    },

    {
        "id": "Q010",
        "question": questions[9],
        "difficulty": "advanced",
        "type": "graph_aggregation",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories",
            "BusinessCheckin",
            "Checkins"
        ],
        "aql": """
LET values = (
    FOR b IN Businesses
        FILTER b.city == "Los Angeles"

        LET categories = (
            FOR c IN 1..1 OUTBOUND b BusinessCategory
                RETURN c.category_name
        )

        FILTER "Moroccan" IN categories
        FILTER "Restaurants" IN categories

        FOR ch IN 1..1 OUTBOUND b BusinessCheckin
            FILTER ch.day == "Friday"
            RETURN ch.count
)

RETURN SUM(values)
"""
    },

    {
        "id": "Q011",
        "question": questions[10],
        "difficulty": "advanced",
        "type": "graph_group_aggregation",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories",
            "BusinessCheckin",
            "Checkins"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Moroccan" IN categories
    FILTER "Restaurants" IN categories

    FOR ch IN 1..1 OUTBOUND b BusinessCheckin
        COLLECT day = ch.day
        AGGREGATE total_checkins = SUM(ch.count)

        SORT day

        RETURN {
            day: day,
            total_checkins: total_checkins
        }
"""
    },

    {
        "id": "Q012",
        "question": questions[11],
        "difficulty": "advanced",
        "type": "graph_group_aggregation",
        "collections": [
            "Businesses",
            "BusinessCategory",
            "Categories",
            "BusinessCheckin",
            "Checkins"
        ],
        "aql": """
FOR b IN Businesses

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Italian" IN categories
    FILTER "Delis" IN categories

    FOR ch IN 1..1 OUTBOUND b BusinessCheckin
        FILTER ch.day == "Sunday"

        COLLECT state = b.state
        AGGREGATE total_checkins = SUM(ch.count)

        SORT state

        RETURN {
            state: state,
            total_checkins: total_checkins
        }
"""
    },

    {
        "id": "Q013",
        "question": questions[12],
        "difficulty": "medium",
        "type": "graph_traversal",
        "collections": [
            "Users",
            "WritesReview",
            "Reviews"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Niloofar"

    FOR r IN 1..1 OUTBOUND u WritesReview

        RETURN DISTINCT {
            review_id: r.rid,
            user_id: r.user_id,
            business_id: r.business_id,
            rating: r.rating,
            text: r.text,
            year: r.year
        }
"""
    },

    {
        "id": "Q014",
        "question": questions[13],
        "difficulty": "advanced",
        "type": "graph_traversal",
        "collections": [
            "Users",
            "WritesReview",
            "Reviews",
            "ReviewsBusiness",
            "Businesses"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Niloofar"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FOR b IN 1..1 OUTBOUND r ReviewsBusiness

            RETURN DISTINCT {
                business_id: b.business_id,
                name: b.name,
                city: b.city,
                state: b.state,
                rating: b.rating
            }
"""
    },

    {
        "id": "Q015",
        "question": questions[14],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Users",
            "WritesReview",
            "Reviews"
        ],
        "aql": """
LET reviews_2015 = (
    FOR u IN Users
        FILTER u.name == "Niloofar"

        FOR r IN 1..1 OUTBOUND u WritesReview
            FILTER r.year == 2015
            RETURN DISTINCT r._id
)

RETURN LENGTH(reviews_2015)
"""
    },

    {
        "id": "Q016",
        "question": questions[15],
        "difficulty": "advanced",
        "type": "graph_traversal_filter",
        "collections": [
            "Users",
            "WritesReview",
            "Reviews",
            "ReviewsBusiness",
            "Businesses"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Niloofar"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FILTER r.rating == 5

        FOR b IN 1..1 OUTBOUND r ReviewsBusiness

            RETURN DISTINCT {
                business_id: b.business_id,
                name: b.name,
                city: b.city,
                state: b.state
            }
"""
    },

    {
        "id": "Q017",
        "question": questions[16],
        "difficulty": "medium",
        "type": "graph_aggregation",
        "collections": [
            "Users",
            "WritesReview",
            "Reviews"
        ],
        "aql": """
LET ratings = (
    FOR u IN Users
        FILTER u.name == "Michelle"

        FOR r IN 1..1 OUTBOUND u WritesReview
            RETURN r.rating
)

RETURN AVERAGE(ratings)
"""
    },

    {
        "id": "Q018",
        "question": questions[17],
        "difficulty": "medium",
        "type": "graph_sort_limit",
        "collections": [
            "Businesses",
            "BusinessCheckin",
            "Checkins"
        ],
        "aql": """
FOR b IN Businesses

    LET total_checkins = SUM(
        FOR ch IN 1..1 OUTBOUND b BusinessCheckin
            RETURN ch.count
    )

    SORT total_checkins DESC
    LIMIT 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        state: b.state,
        total_checkins: total_checkins
    }
"""
    },

    {
        "id": "Q019",
        "question": questions[18],
        "difficulty": "complex",
        "type": "multi_hop_graph",
        "collections": [
            "Users",
            "WritesReview",
            "Reviews",
            "ReviewsBusiness",
            "Businesses",
            "BusinessCategory",
            "Categories"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Michelle"

    FOR r IN 1..1 OUTBOUND u WritesReview

        FOR b IN 1..1 OUTBOUND r ReviewsBusiness

            LET categories = (
                FOR c IN 1..1 OUTBOUND b BusinessCategory
                    RETURN c.category_name
            )

            FILTER "Italian" IN categories
            FILTER "Restaurants" IN categories

            RETURN DISTINCT {
                review_id: r.rid,
                user_id: r.user_id,
                business_id: b.business_id,
                business_name: b.name,
                rating: r.rating,
                text: r.text,
                year: r.year
            }
"""
    },

    {
        "id": "Q020",
        "question": questions[19],
        "difficulty": "medium",
        "type": "graph_aggregation",
        "collections": [
            "Businesses",
            "BusinessCheckin",
            "Checkins"
        ],
        "aql": """
LET values = (
    FOR b IN Businesses
        FILTER b.name == "Cafe Zinho"

        FOR ch IN 1..1 OUTBOUND b BusinessCheckin
            FILTER ch.day == "Friday"
            RETURN ch.count
)

RETURN SUM(values)
"""
    }

])

print(f"Nombre total de Gold AQL : {len(reference_queries)}")

Nombre total de Gold AQL : 20


In [22]:
for ref in reference_queries[8:20]:

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q009 - find the total checkins in Moroccan restaurants in Los Angeles
OK
Nombre de résultats : 1
Exemple : [0]
Q010 - find the total checkins in Moroccan restaurants in Los Angeles on Fridays
OK
Nombre de résultats : 1
Exemple : [0]
Q011 - find the total checkins in Moroccan restaurants in Los Angeles per day
OK
Nombre de résultats : 0
Exemple : []
Q012 - find the total checkins in Italian delis in each state on Sundays
OK
Nombre de résultats : 6
Exemple : [{'state': 'Arizona', 'total_checkins': 1470}, {'state': 'Lothian', 'total_checkins': 20}, {'state': 'Nevada', 'total_checkins': 890}]
Q013 - list all the reviews by Niloofar
OK
Nombre de résultats : 1
Exemple : [{'review_id': 333686, 'user_id': '1Or2rEztlk4AvlOwXhj6SQ', 'business_id': 'VGhMTZyqmGmM4Lrn9i6yuQ', 'rating': 3, 'text': "The food was 5/5 , but the service was really bad, and the food price comparison to any Persian Restaurant I've ever been was so expensive.", 'year': 2016}]
Q014 - list all the businesses which have a rev

In [23]:
reference_queries.extend([

    {
        "id": "Q021",
        "question": questions[20],
        "difficulty": "advanced",
        "type": "multi_hop_graph_count",
        "collections": [
            "Businesses", "Reviews", "ReviewsBusiness"
        ],
        "aql": """
LET reviews = (
    FOR b IN Businesses
        FILTER b.name == "Cafe Zinho"
        FILTER b.state == "Texas"

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            RETURN DISTINCT r._id
)

RETURN LENGTH(reviews)
"""
    },

    {
        "id": "Q022",
        "question": questions[21],
        "difficulty": "medium",
        "type": "graph_sort_limit",
        "collections": [
            "Users", "WritesReview", "Reviews"
        ],
        "aql": """
FOR u IN Users

    LET review_count = LENGTH(
        FOR r IN 1..1 OUTBOUND u WritesReview
            RETURN 1
    )

    SORT review_count DESC
    LIMIT 1

    RETURN {
        user_id: u.user_id,
        name: u.name,
        review_count: review_count
    }
"""
    },

    {
        "id": "Q023",
        "question": questions[22],
        "difficulty": "advanced",
        "type": "multi_hop_graph_count",
        "collections": [
            "Businesses", "ReviewsBusiness",
            "Reviews", "WritesReview", "Users"
        ],
        "aql": """
LET users = (
    FOR b IN Businesses
        FILTER b.name == "Sushi Too"
        FILTER b.city == "Pittsburgh"

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            FOR u IN 1..1 INBOUND r WritesReview
                RETURN DISTINCT u._id
)

RETURN LENGTH(users)
"""
    },

    {
        "id": "Q024",
        "question": questions[23],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET businesses = (
    FOR b IN Businesses
        FILTER b.city == "Edinburgh"

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Pet Groomers"
            RETURN DISTINCT b._id
)

RETURN LENGTH(businesses)
"""
    },

    {
        "id": "Q025",
        "question": questions[24],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET businesses = (
    FOR b IN Businesses
        FILTER b.city == "Edinburgh"

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Pet Groomers"
            RETURN DISTINCT b._id
)

RETURN LENGTH(businesses)
"""
    },

    {
        "id": "Q026",
        "question": questions[25],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET restaurants = (
    FOR b IN Businesses
        FILTER b.city == "Pittsburgh"
        FILTER b.rating == 4.5

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Restaurants"
            RETURN DISTINCT b._id
)

RETURN LENGTH(restaurants)
"""
    },

    {
        "id": "Q027",
        "question": questions[26],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.rating == 5

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Italian" IN categories
    FILTER "Restaurants" IN categories

    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        state: b.state,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q028",
        "question": questions[27],
        "difficulty": "simple",
        "type": "document_count",
        "collections": ["Tips"],
        "aql": """
RETURN LENGTH(
    FOR t IN Tips
        FILTER t.year == 2015
        RETURN 1
)
"""
    },

    {
        "id": "Q029",
        "question": questions[28],
        "difficulty": "medium",
        "type": "graph_aggregation",
        "collections": [
            "Users", "WritesTip", "Tips"
        ],
        "aql": """
LET likes = (
    FOR u IN Users
        FILTER u.name == "Niloofar"

        FOR t IN 1..1 OUTBOUND u WritesTip
            RETURN t.likes
)

RETURN SUM(likes)
"""
    },

    {
        "id": "Q030",
        "question": questions[29],
        "difficulty": "medium",
        "type": "graph_aggregation",
        "collections": [
            "Businesses", "TipsBusiness", "Tips"
        ],
        "aql": """
LET likes = (
    FOR b IN Businesses
        FILTER b.name == "Cafe Zinho"

        FOR t IN 1..1 INBOUND b TipsBusiness
            RETURN t.likes
)

RETURN SUM(likes)
"""
    },

    {
        "id": "Q031",
        "question": questions[30],
        "difficulty": "complex",
        "type": "multi_hop_graph_aggregation",
        "collections": [
            "Users", "WritesTip", "Tips",
            "TipsBusiness", "Businesses"
        ],
        "aql": """
LET likes = (
    FOR u IN Users
        FILTER u.name == "Niloofar"

        FOR t IN 1..1 OUTBOUND u WritesTip
            FOR b IN 1..1 OUTBOUND t TipsBusiness
                FILTER b.name == "Cafe Zinho"
                RETURN t.likes
)

RETURN SUM(likes)
"""
    },

    {
        "id": "Q032",
        "question": questions[31],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Users", "WritesTip", "Tips"
        ],
        "aql": """
LET tips = (
    FOR u IN Users
        FILTER u.name == "Michelle"

        FOR t IN 1..1 OUTBOUND u WritesTip
            FILTER t.year == 2014
            RETURN DISTINCT t._id
)

RETURN LENGTH(tips)
"""
    },

    {
        "id": "Q033",
        "question": questions[32],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Users", "WritesTip", "Tips"
        ],
        "aql": """
LET tips = (
    FOR u IN Users
        FILTER u.name == "Michelle"

        FOR t IN 1..1 OUTBOUND u WritesTip
            FILTER t.month == 4
                OR LOWER(TO_STRING(t.month)) == "april"

            RETURN DISTINCT t._id
)

RETURN LENGTH(tips)
"""
    },

    {
        "id": "Q034",
        "question": questions[33],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET restaurants = (
    FOR b IN Businesses
        FILTER b.state == "Texas"

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Restaurants"
            RETURN DISTINCT b._id
)

RETURN LENGTH(restaurants)
"""
    },

    {
        "id": "Q035",
        "question": questions[34],
        "difficulty": "advanced",
        "type": "graph_group_sort",
        "collections": [
            "Businesses", "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Madison"

    FOR n IN 1..1 OUTBOUND b BusinessNeighborhood

        COLLECT neighborhood = n.neighborhood_name
        WITH COUNT INTO business_count

        SORT business_count DESC
        LIMIT 1

        RETURN {
            neighborhood: neighborhood,
            business_count: business_count
        }
"""
    },

    {
        "id": "Q036",
        "question": questions[35],
        "difficulty": "advanced",
        "type": "multi_relation_graph",
        "collections": [
            "Businesses",
            "BusinessCategory", "Categories",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Madison"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Italian" IN categories
    FILTER "Restaurants" IN categories

    FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
        RETURN DISTINCT n.neighborhood_name
"""
    },

    {
        "id": "Q037",
        "question": questions[36],
        "difficulty": "advanced",
        "type": "multi_relation_graph_filter",
        "collections": [
            "Businesses",
            "BusinessCategory", "Categories",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Madison"
    FILTER b.rating < 2.5

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Italian" IN categories
    FILTER "Restaurants" IN categories

    FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
        RETURN DISTINCT n.neighborhood_name
"""
    },

    {
        "id": "Q038",
        "question": questions[37],
        "difficulty": "complex",
        "type": "multi_relation_graph_sort",
        "collections": [
            "Businesses",
            "BusinessCategory", "Categories",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Madison"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Italian" IN categories
    FILTER "Restaurants" IN categories

    FOR n IN 1..1 OUTBOUND b BusinessNeighborhood

        SORT b.rating DESC
        LIMIT 1

        RETURN {
            neighborhood: n.neighborhood_name,
            business: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q039",
        "question": questions[38],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.rating == 3.5

    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        state: b.state,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q040",
        "question": questions[39],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }
"""
    }

])

print("Nombre total de Gold AQL :", len(reference_queries))

Nombre total de Gold AQL : 40


In [24]:
for ref in reference_queries[20:40]:

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q021 - find the number of reviews written for “Cafe Zinho” restaurants in Texas
OK
Nombre de résultats : 1
Exemple : [0]
Q022 - find the user with the most number of reviews
OK
Nombre de résultats : 1
Exemple : [{'user_id': 'kGgAARL2UmvCcTRfiscjug', 'name': 'J', 'review_count': 1004}]
Q023 - how many users reviewed Sushi Too in Pittsburgh
OK
Nombre de résultats : 1
Exemple : [83]
Q024 - What is the number of pet groomers in Edinburgh
OK
Nombre de résultats : 1
Exemple : [2]
Q025 - How many pet groomers exist in Edinburgh
OK
Nombre de résultats : 1
Exemple : [2]
Q026 - What is the number of restaurants in Pittsburgh rated 4.5
OK
Nombre de résultats : 1
Exemple : [176]
Q027 - List all 5 star Italian restaurants
OK
Nombre de résultats : 36
Exemple : [{'business_id': 'KXzMaehr0oYn9c4gs-FSBw', 'name': 'Ristorante Vino Rosso', 'city': 'Saint-Laurent', 'state': 'Quebec', 'rating': 5}, {'business_id': 'y38C1Xdo78QW-EwN8ys9YA', 'name': 'Amato', 'city': 'Laval', 'state': 'Quebec', 'rating': 5}, 

In [25]:
reference_queries.extend([

    {
        "id": "Q041",
        "question": questions[40],
        "difficulty": "advanced",
        "type": "multi_hop_graph",
        "collections": [
            "Businesses", "BusinessCategory", "Categories",
            "ReviewsBusiness", "Reviews"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.review_count > 100

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Pet Groomers"

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            RETURN DISTINCT {
                review_id: r.rid,
                business_id: b.business_id,
                business_name: b.name,
                rating: r.rating,
                text: r.text,
                year: r.year
            }
"""
    },

    {
        "id": "Q042",
        "question": questions[41],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.review_count > 100

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Pet Groomers"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            review_count: b.review_count,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q043",
        "question": questions[42],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"
    FILTER b.rating >= 4

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Chinese" IN categories
    FILTER "Restaurants" IN categories

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q044",
        "question": questions[43],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"
    FILTER b.rating > 3.5

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Mexican" IN categories
    FILTER "Restaurants" IN categories

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q045",
        "question": questions[44],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Gyms"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q046",
        "question": questions[45],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET dentists = (
    FOR b IN Businesses
        FILTER b.city == "Los Angeles"

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Dentists"

            RETURN DISTINCT b._id
)

RETURN LENGTH(dentists)
"""
    },

    {
        "id": "Q047",
        "question": questions[46],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET dentists = (
    FOR b IN Businesses
        FILTER b.city == "Los Angeles"
        FILTER b.rating > 4

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Dentists"

            RETURN DISTINCT b._id
)

RETURN LENGTH(dentists)
"""
    },

    {
        "id": "Q048",
        "question": questions[47],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Bars"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q049",
        "question": questions[48],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Dance Schools"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q050",
        "question": questions[49],
        "difficulty": "advanced",
        "type": "graph_count",
        "collections": [
            "Businesses", "ReviewsBusiness", "Reviews",
            "WritesReview", "Users"
        ],
        "aql": """
LET users = (
    FOR b IN Businesses
        FILTER b.name == "Texas de Brazil"
        FILTER b.city == "Dallas"
        FILTER b.state == "Texas"

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            FOR u IN 1..1 INBOUND r WritesReview
                RETURN DISTINCT u._id
)

RETURN LENGTH(users)
"""
    },

    {
        "id": "Q051",
        "question": questions[50],
        "difficulty": "advanced",
        "type": "graph_count",
        "collections": [
            "Businesses", "ReviewsBusiness", "Reviews"
        ],
        "aql": """
LET reviews = (
    FOR b IN Businesses
        FILTER b.name == "Bistro Di Napoli"

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            FILTER r.year == 2015
            RETURN DISTINCT r._id
)

RETURN LENGTH(reviews)
"""
    },

    {
        "id": "Q052",
        "question": questions[51],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"
    FILTER b.review_count > 100

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating,
            review_count: b.review_count
        }
"""
    },

    {
        "id": "Q053",
        "question": questions[52],
        "difficulty": "advanced",
        "type": "multi_relation_graph_count",
        "collections": [
            "Businesses",
            "BusinessCategory", "Categories",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
LET restaurants = (
    FOR b IN Businesses
        FILTER b.city == "Dallas"

        LET categories = (
            FOR c IN 1..1 OUTBOUND b BusinessCategory
                RETURN c.category_name
        )

        FILTER "Restaurants" IN categories

        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            FILTER n.neighborhood_name == "Hazelwood"

            RETURN DISTINCT b._id
)

RETURN LENGTH(restaurants)
"""
    },

    {
        "id": "Q054",
        "question": questions[53],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"
    FILTER b.review_count >= 50

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Hotels"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating,
            review_count: b.review_count
        }
"""
    },

    {
        "id": "Q055",
        "question": questions[54],
        "difficulty": "advanced",
        "type": "multi_hop_graph",
        "collections": [
            "Businesses", "ReviewsBusiness",
            "Reviews", "WritesReview", "Users"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Mesa Grill"

    FOR r IN 1..1 INBOUND b ReviewsBusiness
        FOR u IN 1..1 INBOUND r WritesReview

            RETURN DISTINCT {
                user_id: u.user_id,
                name: u.name
            }
"""
    },

    {
        "id": "Q056",
        "question": questions[55],
        "difficulty": "simple",
        "type": "document_count",
        "collections": ["Businesses"],
        "aql": """
RETURN LENGTH(
    FOR b IN Businesses
        FILTER b.name == "Starbucks"
        FILTER b.city == "Dallas"
        FILTER b.state == "Texas"
        RETURN b._id
)
"""
    },

    {
        "id": "Q057",
        "question": questions[56],
        "difficulty": "advanced",
        "type": "graph_count",
        "collections": [
            "Businesses", "ReviewsBusiness", "Reviews"
        ],
        "aql": """
LET reviews = (
    FOR b IN Businesses
        FILTER b.name == "Acacia Cafe"

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            RETURN DISTINCT r._id
)

RETURN LENGTH(reviews)
"""
    },

    {
        "id": "Q058",
        "question": questions[57],
        "difficulty": "advanced",
        "type": "category_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"
    FILTER b.state == "Texas"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Restaurants" IN categories
    FILTER "Valet Services" IN categories

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q059",
        "question": questions[58],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Walmart"
    FILTER b.city == "Los Angeles"

    RETURN DISTINCT b.full_address
"""
    },

    {
        "id": "Q060",
        "question": questions[59],
        "difficulty": "complex",
        "type": "multi_hop_graph_filter",
        "collections": [
            "Users", "WritesReview", "Reviews",
            "ReviewsBusiness", "Businesses",
            "BusinessCategory", "Categories"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Patrick"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FOR b IN 1..1 OUTBOUND r ReviewsBusiness

            FILTER b.city == "Los Angeles"

            FOR c IN 1..1 OUTBOUND b BusinessCategory
                FILTER c.category_name == "Restaurants"

                RETURN DISTINCT {
                    business_id: b.business_id,
                    name: b.name,
                    rating: b.rating
                }
"""
    }

])

print("Nombre total de Gold AQL :", len(reference_queries))

Nombre total de Gold AQL : 60


In [26]:
for ref in reference_queries[40:60]:

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q041 - Find all the reviews for all pet groomers with more than 100 reviews
OK
Nombre de résultats : 534
Exemple : [{'review_id': 756582, 'business_id': 'lBRCMN7wNn-BKUsaOXJRow', 'business_name': "Wag N' Wash Healthy Pet Center", 'rating': 5, 'text': 'We are so glad we found this place. We have had all 4 of our dogs groomed here and they come home looking gorgeous! The staff is friendly and knowledgeable. We just love it here!', 'year': 2016}, {'review_id': 756581, 'business_id': 'lBRCMN7wNn-BKUsaOXJRow', 'business_name': "Wag N' Wash Healthy Pet Center", 'rating': 5, 'text': 'My dog loves this place. Went to give him a bath after a run at the park.everyone was really nice and attentive. Got all the supplies needed more of all he was sure digging the treats.', 'year': 2016}, {'review_id': 756580, 'business_id': 'lBRCMN7wNn-BKUsaOXJRow', 'business_name': "Wag N' Wash Healthy Pet Center", 'rating': 5, 'text': "We took our Scruffy here today for a self-wash.  We have always taken our long

In [27]:
reference_queries.extend([

    {
        "id": "Q061",
        "question": questions[60],
        "difficulty": "advanced",
        "type": "multi_relation_graph_filter",
        "collections": [
            "Businesses",
            "BusinessCategory", "Categories",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Madison"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Italian" IN categories
    FILTER "Restaurants" IN categories

    FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
        FILTER n.neighborhood_name == "Meadowood"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q062",
        "question": questions[61],
        "difficulty": "complex",
        "type": "multi_hop_graph_filter",
        "collections": [
            "Users", "WritesReview", "Reviews",
            "ReviewsBusiness", "Businesses",
            "BusinessCategory", "Categories"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Patrick"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FOR b IN 1..1 OUTBOUND r ReviewsBusiness

            FOR c IN 1..1 OUTBOUND b BusinessCategory
                FILTER c.category_name == "Bars"

                RETURN DISTINCT {
                    business_id: b.business_id,
                    name: b.name,
                    city: b.city,
                    state: b.state,
                    rating: b.rating
                }
"""
    },

    {
        "id": "Q063",
        "question": questions[62],
        "difficulty": "complex",
        "type": "multi_hop_graph_filter",
        "collections": [
            "Users", "WritesReview", "Reviews",
            "ReviewsBusiness", "Businesses",
            "BusinessCategory", "Categories"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Patrick"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FOR b IN 1..1 OUTBOUND r ReviewsBusiness

            FILTER b.rating >= 3

            FOR c IN 1..1 OUTBOUND b BusinessCategory
                FILTER c.category_name == "Bars"

                RETURN DISTINCT {
                    business_id: b.business_id,
                    name: b.name,
                    city: b.city,
                    state: b.state,
                    rating: b.rating
                }
"""
    },

    {
        "id": "Q064",
        "question": questions[63],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"
    FILTER b.review_count >= 30

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Bars"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating,
            review_count: b.review_count
        }
"""
    },

    {
        "id": "Q065",
        "question": questions[64],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"
    FILTER b.review_count >= 30
    FILTER b.rating > 3

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Bars"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating,
            review_count: b.review_count
        }
"""
    },

    {
        "id": "Q066",
        "question": questions[65],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET restaurants = (
    FOR b IN Businesses
        FILTER b.city == "Edinburgh"

        LET categories = (
            FOR c IN 1..1 OUTBOUND b BusinessCategory
                RETURN c.category_name
        )

        FILTER "Egyptian" IN categories
        FILTER "Restaurants" IN categories

        RETURN DISTINCT b._id
)

RETURN LENGTH(restaurants)
"""
    }

])

print("Nombre total de Gold AQL :", len(reference_queries))

Nombre total de Gold AQL : 66


In [28]:
for ref in reference_queries[60:66]:

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q061 - Find all Italian restaurants in the Meadowood neighborhood of Madison
OK
Nombre de résultats : 0
Exemple : []
Q062 - Find all bars reviewed by Patrick
OK
Nombre de résultats : 253
Exemple : [{'business_id': '_KvGddVMM3d2f7NEa_eOtA', 'name': 'Primanti Bros.', 'city': 'Pittsburgh', 'state': 'Pennsylvania', 'rating': 3}, {'business_id': 'NlxvN9wUw14qIS6ledk15Q', 'name': "Oggi's Sports I Brewhouse I Pizza", 'city': 'Glendale', 'state': 'Arizona', 'rating': 3.5}, {'business_id': 'W0TK5K3VLbnNrrzLTaaykw', 'name': 'Fat Tuesday', 'city': 'Las Vegas', 'state': 'Nevada', 'rating': 4.5}]
Q063 - Find all bars reviewed by Patrick with at least 3 stars
OK
Nombre de résultats : 236
Exemple : [{'business_id': '_KvGddVMM3d2f7NEa_eOtA', 'name': 'Primanti Bros.', 'city': 'Pittsburgh', 'state': 'Pennsylvania', 'rating': 3}, {'business_id': 'NlxvN9wUw14qIS6ledk15Q', 'name': "Oggi's Sports I Brewhouse I Pizza", 'city': 'Glendale', 'state': 'Arizona', 'rating': 3.5}, {'business_id': 'W0TK5K3VLbnNrrzLT

In [29]:
reference_queries.extend([

    {
        "id": "Q067",
        "question": questions[66],
        "difficulty": "advanced",
        "type": "graph_group_aggregation",
        "collections": [
            "Businesses", "BusinessCheckin", "Checkins"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.name IN ["Barrio Café", "Barrio Cafe"]

    FOR ch IN 1..1 OUTBOUND b BusinessCheckin

        COLLECT day = ch.day
        AGGREGATE total_checkins = SUM(ch.count)

        COLLECT AGGREGATE average_checkins = AVERAGE(total_checkins)

        RETURN average_checkins
"""
    },

    {
        "id": "Q068",
        "question": questions[67],
        "difficulty": "advanced",
        "type": "graph_filter_sort_limit",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Indian" IN categories
    FILTER "Restaurants" IN categories

    SORT b.rating DESC
    LIMIT 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating,
        review_count: b.review_count
    }
"""
    },

    {
        "id": "Q069",
        "question": questions[68],
        "difficulty": "complex",
        "type": "multi_hop_graph_filter",
        "collections": [
            "Users", "WritesReview", "Reviews",
            "ReviewsBusiness", "Businesses",
            "BusinessCategory", "Categories"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Patrick"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FOR b IN 1..1 OUTBOUND r ReviewsBusiness

            FILTER b.city == "Dallas"

            LET categories = (
                FOR c IN 1..1 OUTBOUND b BusinessCategory
                    RETURN c.category_name
            )

            FILTER "Restaurants" IN categories

            RETURN DISTINCT {
                business_id: b.business_id,
                name: b.name,
                rating: b.rating
            }
"""
    },

    {
        "id": "Q070",
        "question": questions[69],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
LET businesses = (
    FOR b IN Businesses
        FILTER b.city == "Madison"

        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            FILTER n.neighborhood_name == "Stone Meadows"

            RETURN DISTINCT b._id
)

RETURN LENGTH(businesses)
"""
    },

    {
        "id": "Q071",
        "question": questions[70],
        "difficulty": "complex",
        "type": "multi_hop_graph_filter",
        "collections": [
            "Businesses", "TipsBusiness", "Tips",
            "WritesTip", "Users"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.name IN ["Barrio Café", "Barrio Cafe"]

    FOR t IN 1..1 INBOUND b TipsBusiness
        FILTER t.year == 2015

        FOR u IN 1..1 INBOUND t WritesTip

            RETURN DISTINCT {
                user_id: u.user_id,
                name: u.name
            }
"""
    },

    {
        "id": "Q072",
        "question": questions[71],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.state == "Texas"
    FILTER b.rating < 2

    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q073",
        "question": questions[72],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Seafood" IN categories
    FILTER "Restaurants" IN categories

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q074",
        "question": questions[73],
        "difficulty": "advanced",
        "type": "graph_filter_sort_limit",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Indian" IN categories
    FILTER "Restaurants" IN categories

    SORT b.review_count DESC
    LIMIT 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating,
        review_count: b.review_count
    }
"""
    },

    {
        "id": "Q075",
        "question": questions[74],
        "difficulty": "advanced",
        "type": "graph_filter_sort_limit",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER "Italian" IN categories
    FILTER "Restaurants" IN categories

    SORT b.rating DESC
    LIMIT 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating,
        review_count: b.review_count
    }
"""
    },

    {
        "id": "Q076",
        "question": questions[75],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET bars = (
    FOR b IN Businesses
        FILTER b.city == "Dallas"
        FILTER b.rating > 3.5

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Bars"

            RETURN DISTINCT b._id
)

RETURN LENGTH(bars)
"""
    },

    {
        "id": "Q077",
        "question": questions[76],
        "difficulty": "complex",
        "type": "graph_group_count",
        "collections": [
            "Users", "WritesReview", "Reviews"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Michelle"

    FOR r IN 1..1 OUTBOUND u WritesReview

        COLLECT month = r.month
        AGGREGATE business_count = COUNT_DISTINCT(r.business_id)

        SORT month ASC

        RETURN {
            month: month,
            business_count: business_count
        }
"""
    },

    {
        "id": "Q078",
        "question": questions[77],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER LOWER(c.category_name) IN [
            "pet hospice",
            "pet hospices"
        ]

        FILTER b.city == "Pittsburgh"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q079",
        "question": questions[78],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Users", "WritesReview", "Reviews"
        ],
        "aql": """
LET reviews = (
    FOR u IN Users
        FILTER u.name == "Adrienne"

        FOR r IN 1..1 OUTBOUND u WritesReview
            RETURN DISTINCT r._id
)

RETURN LENGTH(reviews)
"""
    },

    {
        "id": "Q080",
        "question": questions[79],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Users", "WritesReview", "Reviews"
        ],
        "aql": """
LET reviews = (
    FOR u IN Users
        FILTER u.name == "Michelle"

        FOR r IN 1..1 OUTBOUND u WritesReview
            FILTER r.year == 2014
            FILTER r.month == 3
                OR LOWER(TO_STRING(r.month)) == "march"

            RETURN DISTINCT r._id
)

RETURN LENGTH(reviews)
"""
    }

])

print("Nombre total de Gold AQL :", len(reference_queries))

Nombre total de Gold AQL : 80


In [30]:
for ref in reference_queries[66:80]:

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q067 - Find the average number of checkins in restaurant Barrio Café per day
OK
Nombre de résultats : 1
Exemple : [643.7142857142857]
Q068 - Which Indian restaurant in Dallas has the highest rating?
OK
Nombre de résultats : 0
Exemple : []
Q069 - Which restaurants in Dallas were reviewed by user Patrick?
OK
Nombre de résultats : 0
Exemple : []
Q070 - How many businesses are there in the Stone Meadows neighborhood in Madison?
OK
Nombre de résultats : 1
Exemple : [8]
Q071 - Find all users who have written tips for Barrio Café in 2015
OK
Nombre de résultats : 17
Exemple : [{'user_id': 'omr0Ckgy0IZv0iyljxG55w', 'name': 'Monserrat'}, {'user_id': 'AV8ZpKdcf3vw8KqJXKqAsQ', 'name': 'Holly'}, {'user_id': '8Xeb5XWukiutl_agyy49-Q', 'name': 'Franchesca'}]
Q072 - Find all businesses in Texas with a rating below 2
OK
Nombre de résultats : 0
Exemple : []
Q073 - Find all restaurants that serve seafood in Los Angeles
OK
Nombre de résultats : 0
Exemple : []
Q074 - Which Indian restaurant in Dallas has th

In [31]:
reference_queries.extend([

    {
        "id": "Q081",
        "question": questions[80],
        "difficulty": "advanced",
        "type": "graph_count_distinct_businesses",
        "collections": ["Users", "WritesReview", "Reviews"],
        "aql": """
LET businesses = (
    FOR u IN Users
        FILTER u.name == "Michelle"

        FOR r IN 1..1 OUTBOUND u WritesReview
            FILTER r.year == 2010
            RETURN DISTINCT r.business_id
)

RETURN LENGTH(businesses)
"""
    },

    {
        "id": "Q082",
        "question": questions[81],
        "difficulty": "complex",
        "type": "multi_hop_graph_count",
        "collections": [
            "Users", "WritesReview", "Reviews",
            "ReviewsBusiness", "Businesses"
        ],
        "aql": """
LET businesses = (
    FOR u IN Users
        FILTER u.name == "Christine"

        FOR r IN 1..1 OUTBOUND u WritesReview
            FILTER r.year == 2010

            FOR b IN 1..1 OUTBOUND r ReviewsBusiness
                FILTER b.city == "San Diego"
                RETURN DISTINCT b._id
)

RETURN LENGTH(businesses)
"""
    },

    {
        "id": "Q083",
        "question": questions[82],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": ["Users", "WritesReview", "Reviews"],
        "aql": """
FOR u IN Users
    FILTER u.name == "Patrick"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FILTER r.rating > 4

        RETURN DISTINCT {
            review_id: r.rid,
            user_id: r.user_id,
            business_id: r.business_id,
            rating: r.rating,
            text: r.text,
            year: r.year
        }
"""
    },

    {
        "id": "Q084",
        "question": questions[83],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": ["Businesses", "BusinessCategory", "Categories"],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Pittsburgh"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Bistros"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q085",
        "question": questions[84],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": ["Businesses", "BusinessCategory", "Categories"],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Bakeries"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q086",
        "question": questions[85],
        "difficulty": "medium",
        "type": "document_filter",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"
    FILTER CONTAINS(LOWER(b.name), "apple")

    RETURN DISTINCT {
        business_id: b.business_id,
        name: b.name,
        full_address: b.full_address
    }
"""
    },

    {
        "id": "Q087",
        "question": questions[86],
        "difficulty": "simple",
        "type": "document_count",
        "collections": ["Businesses"],
        "aql": """
RETURN LENGTH(
    FOR b IN Businesses
        FILTER b.city == "Los Angeles"
        FILTER b.name == "Target"
        RETURN b._id
)
"""
    },

    {
        "id": "Q088",
        "question": questions[87],
        "difficulty": "complex",
        "type": "multi_hop_graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories",
            "ReviewsBusiness", "Reviews",
            "WritesReview", "Users"
        ],
        "aql": """
LET users = (
    FOR b IN Businesses
        FILTER b.city == "Dallas"

        LET categories = (
            FOR c IN 1..1 OUTBOUND b BusinessCategory
                RETURN c.category_name
        )

        FILTER "Irish Pub" IN categories
            OR "Irish Pubs" IN categories

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            FOR u IN 1..1 INBOUND r WritesReview
                RETURN DISTINCT u._id
)

RETURN LENGTH(users)
"""
    },

    {
        "id": "Q089",
        "question": questions[88],
        "difficulty": "advanced",
        "type": "graph_group_aggregation",
        "collections": ["Users", "WritesReview", "Reviews"],
        "aql": """
FOR u IN Users

    LET ratings = (
        FOR r IN 1..1 OUTBOUND u WritesReview
            RETURN r.rating
    )

    FILTER LENGTH(ratings) > 0

    LET avg_rating = AVERAGE(ratings)

    FILTER avg_rating < 3

    RETURN {
        user_id: u.user_id,
        name: u.name,
        average_rating: avg_rating
    }
"""
    },

    {
        "id": "Q090",
        "question": questions[89],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": ["Businesses", "BusinessCategory", "Categories"],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"
    FILTER b.rating > 4.5

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q091",
        "question": questions[90],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": ["Businesses", "BusinessCategory", "Categories"],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Breweries"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q092",
        "question": questions[91],
        "difficulty": "medium",
        "type": "graph_lookup",
        "collections": [
            "Businesses", "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Flat Top Grill"

    FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
        RETURN DISTINCT n.neighborhood_name
"""
    },

    {
        "id": "Q093",
        "question": questions[92],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": ["Users"],
        "aql": """
FOR u IN Users
    FILTER u.name == "Michelle"
    RETURN DISTINCT u.user_id
"""
    },

    {
        "id": "Q094",
        "question": questions[93],
        "difficulty": "advanced",
        "type": "graph_filter",
        "collections": ["Businesses", "TipsBusiness", "Tips"],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Vintner Grill"

    FOR t IN 1..1 INBOUND b TipsBusiness
        FILTER t.likes > 9

        RETURN DISTINCT {
            tip_id: t.tip_id,
            user_id: t.user_id,
            likes: t.likes,
            text: t.text,
            year: t.year
        }
"""
    },

    {
        "id": "Q095",
        "question": questions[94],
        "difficulty": "advanced",
        "type": "graph_filter",
        "collections": ["Businesses", "ReviewsBusiness", "Reviews"],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Kabob Palace"

    FOR r IN 1..1 INBOUND b ReviewsBusiness
        FILTER r.year == 2014

        RETURN DISTINCT {
            review_id: r.rid,
            user_id: r.user_id,
            rating: r.rating,
            text: r.text
        }
"""
    },

    {
        "id": "Q096",
        "question": questions[95],
        "difficulty": "simple",
        "type": "document_aggregation",
        "collections": ["Reviews"],
        "aql": """
LET ratings = (
    FOR r IN Reviews
        FILTER r.year == 2014
        RETURN r.rating
)

RETURN AVERAGE(ratings)
"""
    },

    {
        "id": "Q097",
        "question": questions[96],
        "difficulty": "advanced",
        "type": "graph_count",
        "collections": ["Businesses", "ReviewsBusiness", "Reviews"],
        "aql": """
LET reviews = (
    FOR b IN Businesses
        FILTER b.name == "Vintner Grill"

        FOR r IN 1..1 INBOUND b ReviewsBusiness
            FILTER r.year == 2010
            RETURN DISTINCT r._id
)

RETURN LENGTH(reviews)
"""
    },

    {
        "id": "Q098",
        "question": questions[97],
        "difficulty": "complex",
        "type": "multi_hop_graph_filter",
        "collections": [
            "Businesses", "TipsBusiness", "Tips",
            "WritesTip", "Users"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.city == "Dallas"

    FOR t IN 1..1 INBOUND b TipsBusiness
        FOR u IN 1..1 INBOUND t WritesTip

            RETURN DISTINCT {
                user_id: u.user_id,
                name: u.name
            }
"""
    },

    {
        "id": "Q099",
        "question": questions[98],
        "difficulty": "simple",
        "type": "document_projection",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Whataburger"
    RETURN DISTINCT b.state
"""
    },

    {
        "id": "Q100",
        "question": questions[99],
        "difficulty": "simple",
        "type": "document_projection",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "MGM Grand Buffet"
    RETURN DISTINCT b.city
"""
    }

])

print("Nombre total de Gold AQL :", len(reference_queries))

Nombre total de Gold AQL : 100


In [32]:
for ref in reference_queries[80:100]:

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q081 - How many businesses has Michelle reviewed in 2010?
OK
Nombre de résultats : 1
Exemple : [363]
Q082 - How many businesses in San Diego has Christine reviewed in 2010?
OK
Nombre de résultats : 1
Exemple : [0]
Q083 - Find all reviews by Patrick with a rating above 4
OK
Nombre de résultats : 825
Exemple : [{'review_id': 568161, 'user_id': 'fRscdpeKUb5Z279-sWvfJQ', 'business_id': 'iDYzGVIF1TDWdjHNgNjCVw', 'rating': 5, 'text': 'We sat on the patio at the south location, the staff were very friendly helpful, I loved salmon with vegetables and beans, they were tasty and very clean. I also love the lamb adobo and the Mexican chocolate pie was a revelation.', 'year': 2014}, {'review_id': 539862, 'user_id': '78OtkZH6wtESG3A03U4zqw', 'business_id': 'zMN8UGd1zDEreT58OCdnyg', 'rating': 5, 'text': 'I had to come up to C-Urb alot for work for a year, but I loved this place, made sure to come here once or twice each visit.  The pizza was so freaking great, and it had cheap beer...  Yum!', 'year'

In [34]:
reference_queries.extend([

    {
        "id": "Q101",
        "question": questions[100],
        "difficulty": "simple",
        "type": "document_projection_filter",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "MGM Grand Buffet"
    FILTER b.state == "Texas"
    RETURN DISTINCT b.city
"""
    },

    {
        "id": "Q102",
        "question": questions[101],
        "difficulty": "advanced",
        "type": "multi_relation_graph_count",
        "collections": [
            "Businesses",
            "BusinessNeighborhood", "Neighborhoods",
            "ReviewsBusiness", "Reviews"
        ],
        "aql": """
LET reviews = (
    FOR b IN Businesses

        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            FILTER n.neighborhood_name == "South Summerlin"

            FOR r IN 1..1 INBOUND b ReviewsBusiness
                RETURN DISTINCT r._id
)

RETURN LENGTH(reviews)
"""
    },

    {
        "id": "Q103",
        "question": questions[102],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET businesses = (
    FOR b IN Businesses
        FILTER b.city == "Madison"

        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name IN [
                "Escape Games",
                "Escape Game"
            ]

            RETURN DISTINCT b._id
)

RETURN LENGTH(businesses)
"""
    },

    {
        "id": "Q104",
        "question": questions[103],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Coffee & Tea"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q105",
        "question": questions[104],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.rating >= 4

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Coffee & Tea"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }
"""
    },

    {
        "id": "Q106",
        "question": questions[105],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.is_open == 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        state: b.state,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q107",
        "question": questions[106],
        "difficulty": "simple",
        "type": "document_count",
        "collections": ["Businesses"],
        "aql": """
RETURN LENGTH(
    FOR b IN Businesses
        FILTER b.is_open == 1
        RETURN 1
)
"""
    },

    {
        "id": "Q108",
        "question": questions[107],
        "difficulty": "simple",
        "type": "document_aggregation",
        "collections": ["Businesses"],
        "aql": """
RETURN AVERAGE(
    FOR b IN Businesses
        RETURN b.rating
)
"""
    },

    {
        "id": "Q109",
        "question": questions[108],
        "difficulty": "simple",
        "type": "document_sort_limit",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    SORT b.review_count DESC
    LIMIT 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        review_count: b.review_count,
        rating: b.rating
    }
"""
    },

    {
        "id": "Q110",
        "question": questions[109],
        "difficulty": "simple",
        "type": "document_sort_limit",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    SORT b.rating DESC, b.review_count DESC
    LIMIT 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        rating: b.rating,
        review_count: b.review_count
    }
"""
    },

    {
        "id": "Q111",
        "question": questions[110],
        "difficulty": "medium",
        "type": "document_group_count",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses

    COLLECT city = b.city
    WITH COUNT INTO business_count

    SORT business_count DESC

    RETURN {
        city: city,
        business_count: business_count
    }
"""
    },

    {
        "id": "Q112",
        "question": questions[111],
        "difficulty": "medium",
        "type": "document_group_count",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses

    COLLECT state = b.state
    WITH COUNT INTO business_count

    SORT business_count DESC

    RETURN {
        state: state,
        business_count: business_count
    }
"""
    },

    {
        "id": "Q113",
        "question": questions[112],
        "difficulty": "medium",
        "type": "graph_group_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses

    FOR c IN 1..1 OUTBOUND b BusinessCategory

        COLLECT category = c.category_name
        WITH COUNT INTO business_count

        SORT business_count DESC

        RETURN {
            category: category,
            business_count: business_count
        }
"""
    },

    {
        "id": "Q114",
        "question": questions[113],
        "difficulty": "medium",
        "type": "document_group_aggregation",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses

    COLLECT city = b.city
    AGGREGATE average_rating = AVERAGE(b.rating)

    SORT average_rating DESC

    RETURN {
        city: city,
        average_rating: average_rating
    }
"""
    },

    {
        "id": "Q115",
        "question": questions[114],
        "difficulty": "medium",
        "type": "document_group_aggregation",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses

    COLLECT state = b.state
    AGGREGATE average_rating = AVERAGE(b.rating)

    SORT average_rating DESC

    RETURN {
        state: state,
        average_rating: average_rating
    }
"""
    },

    {
        "id": "Q116",
        "question": questions[115],
        "difficulty": "advanced",
        "type": "graph_group_aggregation",
        "collections": ["Users", "WritesReview", "Reviews"],
        "aql": """
FOR u IN Users

    LET ratings = (
        FOR r IN 1..1 OUTBOUND u WritesReview
            RETURN r.rating
    )

    FILTER LENGTH(ratings) > 0

    LET average_rating = AVERAGE(ratings)

    SORT average_rating DESC

    RETURN {
        user_id: u.user_id,
        name: u.name,
        average_rating: average_rating
    }
"""
    },

    {
        "id": "Q117",
        "question": questions[116],
        "difficulty": "advanced",
        "type": "graph_count",
        "collections": [
            "Users", "WritesReview", "Reviews"
        ],
        "aql": """
LET users = (
    FOR u IN Users

        LET review_count = LENGTH(
            FOR r IN 1..1 OUTBOUND u WritesReview
                RETURN 1
        )

        FILTER review_count > 100

        RETURN DISTINCT u._id
)

RETURN LENGTH(users)
"""
    },

    {
        "id": "Q118",
        "question": questions[117],
        "difficulty": "advanced",
        "type": "graph_sort_limit",
        "collections": [
            "Users", "WritesReview", "Reviews"
        ],
        "aql": """
FOR u IN Users

    LET review_count = LENGTH(
        FOR r IN 1..1 OUTBOUND u WritesReview
            RETURN 1
    )

    SORT review_count DESC
    LIMIT 10

    RETURN {
        user_id: u.user_id,
        name: u.name,
        review_count: review_count
    }
"""
    },

    {
        "id": "Q119",
        "question": questions[118],
        "difficulty": "medium",
        "type": "document_filter",
        "collections": ["Reviews"],
        "aql": """
FOR r IN Reviews
    FILTER r.rating == 5

    RETURN {
        review_id: r.rid,
        user_id: r.user_id,
        business_id: r.business_id,
        rating: r.rating,
        text: r.text,
        year: r.year
    }
"""
    },

    {
        "id": "Q120",
        "question": questions[119],
        "difficulty": "medium",
        "type": "document_filter",
        "collections": ["Reviews"],
        "aql": """
FOR r IN Reviews
    FILTER r.rating == 1

    RETURN {
        review_id: r.rid,
        user_id: r.user_id,
        business_id: r.business_id,
        rating: r.rating,
        text: r.text,
        year: r.year
    }
"""
    },

    {
        "id": "Q121",
        "question": questions[120],
        "difficulty": "medium",
        "type": "document_group_count",
        "collections": ["Reviews"],
        "aql": """
FOR r IN Reviews

    COLLECT year = r.year
    WITH COUNT INTO review_count

    SORT year ASC

    RETURN {
        year: year,
        review_count: review_count
    }
"""
    },

    {
        "id": "Q122",
        "question": questions[121],
        "difficulty": "medium",
        "type": "document_group_count",
        "collections": ["Reviews"],
        "aql": """
FOR r IN Reviews

    COLLECT rating = r.rating
    WITH COUNT INTO review_count

    SORT rating ASC

    RETURN {
        rating: rating,
        review_count: review_count
    }
"""
    },

    {
        "id": "Q123",
        "question": questions[122],
        "difficulty": "advanced",
        "type": "graph_group_count",
        "collections": [
            "Businesses", "ReviewsBusiness", "Reviews"
        ],
        "aql": """
FOR b IN Businesses

    LET review_count = LENGTH(
        FOR r IN 1..1 INBOUND b ReviewsBusiness
            RETURN 1
    )

    FILTER review_count > 0

    SORT review_count DESC

    RETURN {
        business_id: b.business_id,
        name: b.name,
        review_count: review_count
    }
"""
    },

    {
        "id": "Q124",
        "question": questions[123],
        "difficulty": "advanced",
        "type": "graph_group_aggregation",
        "collections": [
            "Businesses", "ReviewsBusiness", "Reviews"
        ],
        "aql": """
FOR b IN Businesses

    LET ratings = (
        FOR r IN 1..1 INBOUND b ReviewsBusiness
            RETURN r.rating
    )

    FILTER LENGTH(ratings) > 0

    LET average_review_rating = AVERAGE(ratings)

    SORT average_review_rating DESC

    RETURN {
        business_id: b.business_id,
        name: b.name,
        average_review_rating: average_review_rating
    }
"""
    },

    {
        "id": "Q125",
        "question": questions[124],
        "difficulty": "advanced",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    FILTER LENGTH(categories) > 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        categories: categories
    }
"""
    },

    {
        "id": "Q126",
        "question": questions[125],
        "difficulty": "advanced",
        "type": "graph_filter",
        "collections": [
            "Businesses",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses

    LET neighborhoods = (
        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            RETURN n.neighborhood_name
    )

    FILTER LENGTH(neighborhoods) > 0

    RETURN {
        business_id: b.business_id,
        name: b.name,
        neighborhoods: neighborhoods
    }
"""
    },

    {
        "id": "Q127",
        "question": questions[126],
        "difficulty": "advanced",
        "type": "multi_relation_graph",
        "collections": [
            "Businesses",
            "BusinessCategory", "Categories",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR b IN Businesses

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    LET neighborhoods = (
        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            RETURN n.neighborhood_name
    )

    FILTER LENGTH(categories) > 0
    FILTER LENGTH(neighborhoods) > 0

    RETURN {
        business_id: b.business_id,
        name: b.name,
        categories: categories,
        neighborhoods: neighborhoods
    }
"""
    },

    {
        "id": "Q128",
        "question": questions[127],
        "difficulty": "complex",
        "type": "multi_relation_graph_summary",
        "collections": [
            "Businesses",
            "BusinessCategory", "Categories",
            "BusinessNeighborhood", "Neighborhoods",
            "ReviewsBusiness", "Reviews"
        ],
        "aql": """
FOR b IN Businesses

    LET categories = (
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    LET neighborhoods = (
        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            RETURN n.neighborhood_name
    )

    LET review_count = LENGTH(
        FOR r IN 1..1 INBOUND b ReviewsBusiness
            RETURN 1
    )

    RETURN {
        business_id: b.business_id,
        name: b.name,
        city: b.city,
        state: b.state,
        rating: b.rating,
        review_count: review_count,
        categories: categories,
        neighborhoods: neighborhoods
    }
"""
    }

])

print("Nombre total de Gold AQL :", len(reference_queries))

Nombre total de Gold AQL : 128


In [39]:
for ref in reference_queries[100:128]:

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR")
        print(e)

Q101 - Find all cities in Texas in which there is a restaurant callled MGM Grand Buffet
OK
Nombre de résultats : 0
Exemple : []
Q102 - Find the number of reviews on businesses located in South Summerlin neighborhood
OK
Nombre de résultats : 1
Exemple : [3851]
Q103 - How many escape games are there in Madison?
OK
Nombre de résultats : 1
Exemple : [3]
Q104 - Find the users who have given tips on pet groomers
OK
Nombre de résultats : 2399
Exemple : [{'business_id': 'p8GMdsCCy-NV_wZuDaU2Jw', 'name': 'Aroma Cafe', 'city': 'Champaign', 'state': 'Illinois', 'rating': 4}, {'business_id': 'aRkYtXfmEKYG-eTDf_qUsw', 'name': 'Lux Central', 'city': 'Phoenix', 'state': 'Arizona', 'rating': 4.5}, {'business_id': 'exNRVWeo9dBaIEyluD63DQ', 'name': 'Village Coffee Roastery', 'city': 'Scottsdale', 'state': 'Arizona', 'rating': 4.5}]
Q105 - List all reviews for bistros with rating less than 1.5
OK
Nombre de résultats : 1293
Exemple : [{'business_id': 'p8GMdsCCy-NV_wZuDaU2Jw', 'name': 'Aroma Cafe', 'city':

In [40]:
reference_queries = [
    q for q in reference_queries
    if int(q["id"][1:]) <= 103
]

print(len(reference_queries))

103


In [44]:
reference_queries.extend([

    # Q104
    {
        "id": "Q104",
        "question": questions[103],
        "difficulty": "complex",
        "type": "multi_hop_graph",
        "collections": [
            "Users", "WritesTip", "Tips",
            "TipsBusiness", "Businesses",
            "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Pet Groomers"

        FOR t IN 1..1 INBOUND b TipsBusiness
            FOR u IN 1..1 INBOUND t WritesTip

                RETURN DISTINCT {
                    user_id: u.user_id,
                    name: u.name
                }
"""
    },

    # Q105
    {
        "id": "Q105",
        "question": questions[104],
        "difficulty": "complex",
        "type": "multi_hop_graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories",
            "ReviewsBusiness", "Reviews"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.rating < 1.5

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Bistros"

        FOR r IN 1..1 INBOUND b ReviewsBusiness

            RETURN DISTINCT {
                review_id: r.rid,
                user_id: r.user_id,
                business_id: r.business_id,
                rating: r.rating,
                text: r.text,
                year: r.year
            }
"""
    },

    # Q106
    {
        "id": "Q106",
        "question": questions[105],
        "difficulty": "simple",
        "type": "document_count",
        "collections": ["Users"],
        "aql": """
RETURN LENGTH(
    FOR u IN Users
        FILTER u.name == "Michelle"
        RETURN u._id
)
"""
    },

    # Q107
    {
        "id": "Q107",
        "question": questions[106],
        "difficulty": "simple",
        "type": "document_projection",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.rating < 1.5
    RETURN DISTINCT b.city
"""
    },

    # Q108
    {
        "id": "Q108",
        "question": questions[107],
        "difficulty": "complex",
        "type": "multi_hop_graph",
        "collections": [
            "Users", "WritesReview", "Reviews",
            "ReviewsBusiness", "Businesses",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
FOR u IN Users
    FILTER u.name == "Michelle"

    FOR r IN 1..1 OUTBOUND u WritesReview
        FOR b IN 1..1 OUTBOUND r ReviewsBusiness
            FOR n IN 1..1 OUTBOUND b BusinessNeighborhood

                RETURN DISTINCT n.neighborhood_name
"""
    },

    # Q109
    {
        "id": "Q109",
        "question": questions[108],
        "difficulty": "advanced",
        "type": "graph_group_sort_limit",
        "collections": [
            "Reviews", "ReviewsBusiness", "Businesses"
        ],
        "aql": """
FOR r IN Reviews
    FILTER r.month == "April"
        OR r.month == 4

    FOR b IN 1..1 OUTBOUND r ReviewsBusiness

        COLLECT business_key = b._id
        WITH COUNT INTO april_reviews

        SORT april_reviews DESC
        LIMIT 1

        LET business = DOCUMENT(business_key)

        RETURN {
            business_id: business.business_id,
            name: business.name,
            april_reviews: april_reviews
        }
"""
    },

    # Q110
    {
        "id": "Q110",
        "question": questions[109],
        "difficulty": "advanced",
        "type": "graph_filter",
        "collections": [
            "Businesses", "TipsBusiness", "Tips"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Cafe Zinho"
    FILTER b.state == "Texas"

    FOR t IN 1..1 INBOUND b TipsBusiness

        RETURN DISTINCT {
            tip_id: t.tip_id,
            user_id: t.user_id,
            likes: t.likes,
            text: t.text,
            year: t.year
        }
"""
    },

    # Q111
    {
        "id": "Q111",
        "question": questions[110],
        "difficulty": "simple",
        "type": "document_projection",
        "collections": ["Businesses"],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Taj Mahal"
    RETURN DISTINCT b.city
"""
    },

    # Q112
    {
        "id": "Q112",
        "question": questions[111],
        "difficulty": "medium",
        "type": "graph_filter",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.state == "Pennsylvania"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }
"""
    },

    # Q113
    {
        "id": "Q113",
        "question": questions[112],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
LET businesses = (
    FOR b IN Businesses
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Restaurants"
            RETURN DISTINCT b._id
)

RETURN LENGTH(businesses)
"""
    },

    # Q114
    {
        "id": "Q114",
        "question": questions[113],
        "difficulty": "complex",
        "type": "multi_hop_graph",
        "collections": [
            "Businesses", "BusinessCategory", "Categories",
            "ReviewsBusiness", "Reviews",
            "WritesReview", "Users"
        ],
        "aql": """
FOR b IN Businesses

    LET is_restaurant = LENGTH(
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Restaurants"
            LIMIT 1
            RETURN 1
    )

    FILTER is_restaurant > 0

    FOR r IN 1..1 INBOUND b ReviewsBusiness
        FOR u IN 1..1 INBOUND r WritesReview

            RETURN DISTINCT {
                user_id: u.user_id,
                name: u.name
            }
"""
    },

    # Q115
    {
        "id": "Q115",
        "question": questions[114],
        "difficulty": "simple",
        "type": "document_count_distinct",
        "collections": ["Businesses"],
        "aql": """
LET cities = (
    FOR b IN Businesses
        FILTER b.name == "Panda Express"
        RETURN DISTINCT b.city
)

RETURN LENGTH(cities)
"""
    },

    # Q116
    {
        "id": "Q116",
        "question": questions[115],
        "difficulty": "advanced",
        "type": "graph_filter",
        "collections": [
            "Businesses", "TipsBusiness", "Tips"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.name == "Cafe Zinho"
    FILTER b.state == "Pennsylvania"

    FOR t IN 1..1 INBOUND b TipsBusiness
        FILTER t.year == 2010

        RETURN DISTINCT {
            tip_id: t.tip_id,
            user_id: t.user_id,
            likes: t.likes,
            text: t.text
        }
"""
    },

    # Q117
    {
        "id": "Q117",
        "question": questions[116],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": ["Users", "WritesTip", "Tips"],
        "aql": """
LET tips = (
    FOR u IN Users
        FILTER u.name == "Michelle"

        FOR t IN 1..1 OUTBOUND u WritesTip
            RETURN DISTINCT t._id
)

RETURN LENGTH(tips)
"""
    },

    # Q118
    {
        "id": "Q118",
        "question": questions[117],
        "difficulty": "medium",
        "type": "graph_count",
        "collections": ["Users", "WritesTip", "Tips"],
        "aql": """
LET tips = (
    FOR u IN Users
        FILTER u.name == "Michelle"

        FOR t IN 1..1 OUTBOUND u WritesTip
            FILTER t.year == 2010
            RETURN DISTINCT t._id
)

RETURN LENGTH(tips)
"""
    },

    # Q119
    {
        "id": "Q119",
        "question": questions[118],
        "difficulty": "complex",
        "type": "multi_hop_graph",
        "collections": [
            "Businesses", "BusinessCategory", "Categories",
            "ReviewsBusiness", "Reviews",
            "WritesReview", "Users"
        ],
        "aql": """
FOR b IN Businesses

    LET is_restaurant = LENGTH(
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            FILTER c.category_name == "Restaurants"
            LIMIT 1
            RETURN 1
    )

    FILTER is_restaurant > 0

    FOR r IN 1..1 INBOUND b ReviewsBusiness
        FILTER r.year == 2010

        FOR u IN 1..1 INBOUND r WritesReview

            RETURN DISTINCT {
                user_id: u.user_id,
                name: u.name
            }
"""
    },

    # Q120
    {
        "id": "Q120",
        "question": questions[119],
        "difficulty": "advanced",
        "type": "graph_sum",
        "collections": [
            "Businesses",
            "BusinessNeighborhood", "Neighborhoods",
            "BusinessCheckin", "Checkins"
        ],
        "aql": """
LET values = (
    FOR b IN Businesses

        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            FILTER n.neighborhood_name == "Brighton Heights"

            FOR ch IN 1..1 OUTBOUND b BusinessCheckin
                RETURN ch.count
)

RETURN SUM(values)
"""
    },

    # Q121
    {
        "id": "Q121",
        "question": questions[120],
        "difficulty": "simple",
        "type": "document_sum",
        "collections": ["Checkins"],
        "aql": """
LET values = (
    FOR ch IN Checkins
        FILTER ch.day == "Sunday"
        RETURN ch.count
)

RETURN SUM(values)
"""
    },

    # Q122
    {
        "id": "Q122",
        "question": questions[121],
        "difficulty": "complex",
        "type": "multi_relation_graph",
        "collections": [
            "Users", "WritesReview", "Reviews",
            "WritesTip", "Tips"
        ],
        "aql": """
FOR u IN Users

    LET reviewed_in_2012 = LENGTH(
        FOR r IN 1..1 OUTBOUND u WritesReview
            FILTER r.year == 2012
            LIMIT 1
            RETURN 1
    )

    FILTER reviewed_in_2012 > 0

    FOR t IN 1..1 OUTBOUND u WritesTip

        RETURN DISTINCT {
            tip_id: t.tip_id,
            user_id: t.user_id,
            business_id: t.business_id,
            likes: t.likes,
            text: t.text,
            year: t.year
        }
"""
    },

    # Q123
    {
        "id": "Q123",
        "question": questions[122],
        "difficulty": "simple",
        "type": "document_count",
        "collections": ["Reviews"],
        "aql": """
RETURN LENGTH(
    FOR r IN Reviews
        FILTER r.month == "March"
            OR r.month == 3
        RETURN r._id
)
"""
    },

    # Q124
    {
        "id": "Q124",
        "question": questions[123],
        "difficulty": "medium",
        "type": "document_group_count",
        "collections": ["Tips"],
        "aql": """
FOR t IN Tips

    COLLECT month = t.month
    WITH COUNT INTO tip_count

    SORT month ASC

    RETURN {
        month: month,
        tip_count: tip_count
    }
"""
    },

    # Q125
    {
        "id": "Q125",
        "question": questions[124],
        "difficulty": "advanced",
        "type": "graph_count_distinct",
        "collections": [
            "Businesses",
            "BusinessNeighborhood", "Neighborhoods"
        ],
        "aql": """
LET neighborhoods = (
    FOR b IN Businesses
        FILTER b.city == "Madison"
        FILTER b.rating == 5

        FOR n IN 1..1 OUTBOUND b BusinessNeighborhood
            RETURN DISTINCT n.neighborhood_name
)

RETURN LENGTH(neighborhoods)
"""
    },

    # Q126
    {
        "id": "Q126",
        "question": questions[125],
        "difficulty": "advanced",
        "type": "graph_sort_limit",
        "collections": [
            "Businesses", "BusinessCategory", "Categories"
        ],
        "aql": """
FOR b IN Businesses

    LET categories = UNIQUE(
        FOR c IN 1..1 OUTBOUND b BusinessCategory
            RETURN c.category_name
    )

    LET category_count = LENGTH(categories)

    SORT category_count DESC
    LIMIT 1

    RETURN {
        business_id: b.business_id,
        name: b.name,
        category_count: category_count,
        categories: categories
    }
"""
    },

    # Q127
    {
        "id": "Q127",
        "question": questions[126],
        "difficulty": "simple",
        "type": "document_filter",
        "collections": ["Reviews"],
        "aql": """
FOR r IN Reviews
    FILTER r.rating < 1

    RETURN {
        review_id: r.rid,
        user_id: r.user_id,
        business_id: r.business_id,
        rating: r.rating,
        text: r.text
    }
"""
    },

    # Q128
    {
        "id": "Q128",
        "question": questions[127],
        "difficulty": "advanced",
        "type": "graph_filter",
        "collections": [
            "Businesses", "ReviewsBusiness", "Reviews"
        ],
        "aql": """
FOR b IN Businesses
    FILTER b.rating == 2.5

    FOR r IN 1..1 INBOUND b ReviewsBusiness

        RETURN DISTINCT {
            review_id: r.rid,
            user_id: r.user_id,
            business_id: r.business_id,
            rating: r.rating,
            text: r.text,
            year: r.year
        }
"""
    }

])

print("Nombre total de Gold AQL :", len(reference_queries))

Nombre total de Gold AQL : 128


In [45]:
for ref in reference_queries[103:128]:
    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")
        print("Nombre de résultats :", len(result))
        print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR :", e)

Q104 - Find the users who have given tips on pet groomers
OK
Nombre de résultats : 1095
Exemple : [{'user_id': 'A_O8wZOsMTPwyeYA4-Rsow', 'name': 'Dave'}, {'user_id': 'AYGHNy8gPxl2Q-etTT3hZw', 'name': 'Anthony'}, {'user_id': 'a0EhgS-pvrVFvRouL4cNFQ', 'name': 'Phil'}]
Q105 - List all reviews for bistros with rating less than 1.5
OK
Nombre de résultats : 0
Exemple : []
Q106 - Find the number of users called Michelle
OK
Nombre de résultats : 1
Exemple : [3700]
Q107 - Find the cities of businesses rated below 1.5
OK
Nombre de résultats : 70
Exemple : ['Phoenix', 'Madison', 'Scottsdale']
Q108 - In which neighborhoods Michelle has reviewed a buissness?
OK
Nombre de résultats : 95
Exemple : ['The Strip', 'Spring Valley', 'Eastside']
Q109 - Find the business with the most number of reviews in April
OK
Nombre de résultats : 1
Exemple : [{'business_id': '4bEjOyTaDG24SY5TxsaUNQ', 'name': 'Mon Ami Gabi', 'april_reviews': 463}]
Q110 - Find all tips for "Cafe Zinho" in Texas.
OK
Nombre de résultats :

In [46]:
reference_queries[113]["aql"] = """
FOR c IN Categories
    FILTER c.category_name == "Restaurants"

    FOR b IN 1..1 INBOUND c BusinessCategory

        FOR r IN 1..1 INBOUND b ReviewsBusiness

            FOR u IN 1..1 INBOUND r WritesReview

                COLLECT user_id = u.user_id, name = u.name

                RETURN {
                    user_id: user_id,
                    name: name
                }
"""

In [47]:
reference_queries[121]["aql"] = """
LET users_2012 = (
    FOR r IN Reviews
        FILTER r.year == 2012

        FOR u IN 1..1 INBOUND r WritesReview
            COLLECT user_key = u._id
            RETURN user_key
)

FOR user_key IN users_2012

    FOR t IN 1..1 OUTBOUND user_key WritesTip

        COLLECT tip_key = t._id

        LET tip = DOCUMENT(tip_key)

        RETURN {
            tip_id: tip.tip_id,
            user_id: tip.user_id,
            business_id: tip.business_id,
            likes: tip.likes,
            text: tip.text,
            year: tip.year
        }
"""

In [48]:
for qid in ["Q114", "Q122"]:

    ref = next(q for q in reference_queries if q["id"] == qid)

    print("=" * 70)
    print(ref["id"], "-", ref["question"])

    try:
        result = list(db.aql.execute(ref["aql"]))

        print("OK")

        if len(result) == 1 and isinstance(result[0], (int, float)):
            print("Réponse :", result[0])
        else:
            print("Nombre de résultats :", len(result))
            print("Exemple :", result[:3])

    except Exception as e:
        print("ERREUR :", e)

Q114 - List all users who reviewed businesses that are restaurants.
OK
Nombre de résultats : 267784
Exemple : [{'user_id': '__0gFbxCoGhySfOfjP--Mg', 'name': 'Johnny'}, {'user_id': '__1ABiFDu7NVVUEU2KJCwQ', 'name': 'Kayayu'}, {'user_id': '__26EDQ1FacBdY4gChHsuA', 'name': 'Michael'}]
Q122 - Find all the tips from a user who has written a review in 2012
OK
Nombre de résultats : 219326
Exemple : [{'tip_id': 642745, 'user_id': 'f_QwW8uaXLQXJXaCYMcCYA', 'business_id': 'e04o1y0ANz7X17-shFCVNw', 'likes': 0, 'text': "Getting my wife's favorite macadamia chocolate candy.", 'year': 2013}, {'tip_id': 642752, 'user_id': 'usX2N3dfDKX0gtTgRufbUA', 'business_id': 'e04o1y0ANz7X17-shFCVNw', 'likes': 0, 'text': '$1 one liter Nestle Pure Life water... Best deal in town! (Reg. $1.75)', 'year': 2015}, {'tip_id': 642767, 'user_id': 'Uchznb_3tmqIpJTh16iRPg', 'business_id': 'kzPMaCsccpwTbLvT5IsRIg', 'likes': 0, 'text': 'Come here early it gets packed', 'year': 2015}]


In [49]:
import json

benchmark = []

for q in reference_queries:
    benchmark.append({
        "id": q["id"],
        "question": q["question"],
        "gold_aql": q["aql"],
        "intent": q.get("intent", ""),
        "difficulty": q.get("difficulty", ""),
        "type": q.get("type", ""),
        "collections": q.get("collections", []),
        "attributes": q.get("attributes", []),
        "comparison_method": q.get("comparison_method", "")
    })

assert len(benchmark) == 128, f"Erreur : {len(benchmark)} questions au lieu de 128"

with open(BENCHMARK_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        benchmark,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Benchmark sauvegardé avec succès.")
print("Nombre de questions :", len(benchmark))
print("Fichier :", BENCHMARK_OUTPUT_PATH)

Benchmark sauvegardé avec succès.
Nombre de questions : 128
Fichier : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\data\benchmark\yelp_benchmark.json
